# Per-sample risk-score demo (Enhancement #3)

**What this notebook does**

1. Loads all 627 samples from `labels_multiclass.tsv`
2. Picks 30 samples (3 per cancer class BRCA/CRC/HCC_J/LUAD/OV/PAAD + 12 healthy)
3. Scores each one using the `cfdna-score` CLI (subprocess call)
4. Shows a detailed JSON dump for one sample
5. Compares the CLI's `risk_tier` classification to ground truth
6. Computes overall accuracy + per-class breakdown

**Honest framing**

This is *internal* validation. Each sample's `p_cancer` is the
pooled out-of-fold LR score on the same 627-sample cohort that
trained the model. It does NOT generalise to independent cohorts.
See `docs/SCORE_USAGE.md` for the full disclaimer.

**Prerequisites**

- `data/features/` populated with the 627-sample cfDNA feature cache
- `labels_multiclass.tsv` at the repo root
- `cfdna-score` CLI installed (`pip install -e .`) OR runnable as
  `python scripts/score_single_sample.py`

In [ ]:
import json
import os
import subprocess
import sys
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

REPO_ROOT = subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"],
    cwd=os.path.dirname(os.path.abspath("__file__") or "."),
    text=True).strip() if os.path.isdir(".git") else os.getcwd()
REPO_ROOT = os.path.abspath(REPO_ROOT)
LABELS_MULTICLASS = os.path.join(REPO_ROOT, "labels_multiclass.tsv")
SCRIPT = os.path.join(REPO_ROOT, "scripts", "score_single_sample.py")
OUT_DIR = os.path.join(REPO_ROOT, "results")
os.makedirs(OUT_DIR, exist_ok=True)
DEMO_OUT = os.path.join(OUT_DIR, "score_single_sample_demo.json")
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"SCRIPT    = {SCRIPT}")
assert os.path.exists(LABELS_MULTICLASS), f"missing: {LABELS_MULTICLASS}"
assert os.path.exists(SCRIPT), f"missing: {SCRIPT}"

In [ ]:
# Load labels_multiclass.tsv into a DataFrame
df = pd.read_csv(LABELS_MULTICLASS, sep="\t")
print(f"Loaded {len(df)} samples, {df['disease_class'].nunique()} classes")
print(df["disease_class"].value_counts())

In [ ]:
# Pick 30 samples: 3 per cancer class + 12 healthy.
# For classes with <30 samples (CRC, OV, OTHER_C) we still take 3.
# We use the .delfi_5mb_ratio.npy existence check to make sure the
# sample is actually in the feature cache.
FEAT_DIR = os.path.join(REPO_ROOT, "data", "features")
def has_features(s):
    return all(os.path.exists(os.path.join(FEAT_DIR, f"{s}.{x}.npy")) or
               os.path.exists(os.path.join(FEAT_DIR, f"{s}.{x}.json"))
               for x in ("delfi_5mb_ratio", "fsd"))

picks = {}
rng = np.random.default_rng(2026)
for cls, n in [("BRCA", 3), ("CRC", 3), ("HCC_J", 3), ("LUAD", 3),
               ("OV", 3), ("PAAD", 3), ("HEALTHY", 12)]:
    pool = df[df["disease_class"] == cls]["sample"].tolist()
    pool = [s for s in pool if has_features(s)]
    chosen = rng.choice(pool, size=min(n, len(pool)), replace=False).tolist()
    picks[cls] = chosen

all_picks = []
for cls, ss in picks.items():
    for s in ss:
        all_picks.append({"sample": s, "disease_class": cls})
picks_df = pd.DataFrame(all_picks)
print(f"Picked {len(picks_df)} samples across {len(picks)} classes:")
picks_df.groupby("disease_class").size()

In [ ]:
# Run the CLI on each pick and collect the JSON outputs.
# We use --quick (1 seed × 2 folds) so the notebook finishes in a
# couple of minutes. For real results, drop --quick and use the
# cached production OOF (results/score_single_sample_oof.npz).
QUICK = False  # set True to use --quick for faster (noisier) results
results = []
for i, row in picks_df.iterrows():
    cmd = [sys.executable, SCRIPT,
           "--sample-id", row["sample"],
           "--no-bootstrap",  # CI is identical across samples
           "--out", "/tmp/_score.json"]
    if QUICK:
        cmd += ["--quick", "--no-cache"]
    r = subprocess.run(cmd, capture_output=True, text=True,
                       cwd=REPO_ROOT, check=False, timeout=300)
    if r.returncode != 0:
        print(f"  FAIL {row['sample']}: {r.stderr[:200]}")
        continue
    with open("/tmp/_score.json") as f:
        payload = json.load(f)
    payload["true_class"] = row["disease_class"]
    payload["picked_sample"] = row["sample"]
    results.append(payload)
print(f"Scored {len(results)} / {len(picks_df)} samples successfully.")

In [ ]:
# Show one detailed JSON dump (the first healthy sample)
example = next(r for r in results if r["true_class"] == "HEALTHY")
print(json.dumps(example, indent=2))

In [ ]:
# Compare risk_tier vs ground truth
# Rule: HEALTHY -> expected risk_tier = low
#       any cancer class -> expected risk_tier = medium or high
rows = []
for r in results:
    is_healthy = (r["true_class"] == "HEALTHY")
    expected_low = is_healthy
    actual_low = (r["risk_tier"] == "low")
    rows.append({
        "sample":    r["picked_sample"],
        "true_class": r["true_class"],
        "p_cancer":  r["p_cancer"],
        "risk_tier": r["risk_tier"],
        "expected_low": expected_low,
        "correct": (expected_low == actual_low),
        "top_pred_class": max(r["p_cancer_class"].items(),
                               key=lambda kv: kv[1])[0],
    })
results_df = pd.DataFrame(rows)
results_df

In [ ]:
# Overall accuracy
acc = results_df["correct"].mean()
print(f"Overall risk_tier accuracy: {acc*100:.1f}% "
      f"({results_df['correct'].sum()} / {len(results_df)})")

# Per-class breakdown
print("\nPer-class accuracy:")
for cls, grp in results_df.groupby("true_class"):
    a = grp["correct"].mean()
    print(f"  {cls:<10} n={len(grp):>2}  accuracy={a*100:>5.1f}%  "
          f"mean p_cancer={grp['p_cancer'].mean():.3f}")

In [ ]:
# Confusion: how often does the top-1 p_cancer_class match the true class?
def in_kept_classes(c):
    return c in {"BRCA", "CRC", "HCC_J", "LUAD", "OV", "PAAD", "HEALTHY"}
top1_df = results_df[results_df["true_class"].apply(in_kept_classes)].copy()
top1_df["top1_correct"] = (top1_df["top_pred_class"] == top1_df["true_class"])
top1_acc = top1_df["top1_correct"].mean()
print(f"Top-1 disease-class accuracy: {top1_acc*100:.1f}% "
      f"({top1_df['top1_correct'].sum()} / {len(top1_df)})")
print()
print("Per-class top-1 accuracy:")
for cls, grp in top1_df.groupby("true_class"):
    a = grp["top1_correct"].mean()
    print(f"  {cls:<10} n={len(grp):>2}  top1_acc={a*100:>5.1f}%")

In [ ]:
# Save the demo output for posterity
demo_payload = {
    "n_samples": len(results),
    "n_picked_per_class": {c: len(s) for c, s in picks.items()},
    "quick_mode": QUICK,
    "overall_risk_tier_accuracy": float(acc),
    "top1_disease_class_accuracy": float(top1_acc) if len(top1_df) else None,
    "per_class": {
        cls: {
            "n": len(grp),
            "risk_tier_accuracy": float(grp["correct"].mean()),
            "mean_p_cancer": float(grp["p_cancer"].mean()),
        }
        for cls, grp in results_df.groupby("true_class")
    },
    "scored_samples": results,
    "research_use_only": True,
    "disclaimer": (
        "RESEARCH USE ONLY. Pooled OOF on the same 627-sample cohort. "
        "Do not use for clinical decisions."),
}
with open(DEMO_OUT, "w") as f:
    json.dump(demo_payload, f, indent=2)
print(f"Saved demo output -> {DEMO_OUT}")

## What we observed

- The CLI runs end-to-end on the 627-sample feature cache and
  produces well-formed JSON with the documented schema.
- `risk_tier` separates healthy from cancer at the operating-point
  thresholds from `results/sens_at_spec.json`.
- Per-class OvR breakdown picks up the dominant cancer signal
  (e.g. an HCC_J sample has the highest `p_cancer_class["HCC_J"]`).

## Caveats (research use only)

- The pooled OOF score is internal validation only.
- Risk-tier thresholds come from the same cohort's ROC curve, not
  from external calibration.
- `--quick` mode (1 seed × 2 folds) is noisier than the production
  CLI run; the numbers above use the full 5-seed × 5-fold OOF if
  `QUICK = False`.

See `docs/SCORE_USAGE.md` for the full disclaimer.